In [ ]:
!pip install pydantic requests firecrawl-py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.5/238.5 kB 7.7 MB/s eta 0:00:00


In [4]:
from google.colab import files
files.upload()

Saving vending-machines-formateado.csv to vending-machines-formateado (1).csv


{'vending-machines-formateado (1).csv': b'Fabricante,Modelo,URL\r\nAzkoyen,Vitro X5 Touch,https://azkoyenvending.com/vitro/\r\nAzkoyen,Vitro X5,https://azkoyenvending.com/vitro/\r\nAzkoyen,Vitro X1 Espresso,https://azkoyenvending.com/vitro/\r\nAzkoyen,Vitro X1 MIA,https://azkoyenvending.com/vitro/\r\nAzkoyen,Vitro S5 Espresso,https://azkoyenvending.com/vitro/\r\nAzkoyen,Vitro S5 Instant,https://azkoyenvending.com/vitro/\r\nAzkoyen,Vitro S5 MIA,https://azkoyenvending.com/vitro/\r\nAzkoyen,Vitro S1 Espresso,https://azkoyenvending.com/vitro/\r\nAzkoyen,Vitro S1 Instant,https://azkoyenvending.com/vitro/\r\nAzkoyen,Vitro S2 Instant,https://azkoyenvending.com/vitro/\r\nAzkoyen,Vitro S3 FBT,https://azkoyenvending.com/vitro/\r\nAzkoyen,Vitro S4 Instant,https://azkoyenvending.com/vitro/\r\nAzkoyen,Vitro X3 Duo,https://azkoyenvending.com/vitro/\r\nAzkoyen,Vitro X3 Espresso,https://azkoyenvending.com/vitro/\r\nAzkoyen,Vitro X4 Duo,https://azkoyenvending.com/vitro/\r\nAzkoyen,Vitro X4 Espresso,htt

In [ ]:
!pip uninstall -y firecrawl firecrawl-py
!pip install -U firecrawl pydantic requests

Found existing installation: firecrawl 4.28.0
Uninstalling firecrawl-4.28.0:
  Successfully uninstalled firecrawl-4.28.0


In [ ]:
import os

DATABASE_NAME = "vending_api_data.db"

if os.path.exists(DATABASE_NAME):
    os.remove(DATABASE_NAME)
    print(f"Base eliminada: {DATABASE_NAME}")
else:
    print("No había base previa.")

Base eliminada: vending_api_data.db


In [ ]:
# ============================================================
# GOOGLE COLAB - PIPELINE COMPLETO DE EXTRACCIÓN VENDING
# Firecrawl + Serper + SQLite
# Continúa desde el registro 233 en adelante
# ============================================================

# ------------------------------------------------------------
# 0. INSTALACIÓN AUTOMÁTICA DE DEPENDENCIAS EN COLAB
# ------------------------------------------------------------
import sys
import subprocess
import importlib.util

def install_if_missing(package_name: str, import_name: str = None):
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        print(f"[*] Instalando {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", package_name])

install_if_missing("firecrawl-py", "firecrawl")
install_if_missing("pydantic", "pydantic")
install_if_missing("requests", "requests")
install_if_missing("beautifulsoup4", "bs4")
install_if_missing("pandas", "pandas")


# ------------------------------------------------------------
# 1. IMPORTS
# ------------------------------------------------------------
import os
import json
import time
import sqlite3
import requests
from typing import List, Optional, Any, Dict
from urllib.parse import urljoin

import pandas as pd
from bs4 import BeautifulSoup
from pydantic import BaseModel, Field

from firecrawl import Firecrawl


# ------------------------------------------------------------
# 2. CONFIGURACIÓN Y CREDENCIALES
# ------------------------------------------------------------

# PEGA AQUÍ TUS CLAVES REALES
SERPER_API_KEY = "7533196408e54b51b43d4a297eb79c7dafa1f2c0"
FIRECRAWL_API_KEY = "fc-8dd928833e154c4caaf705030d6a1af3"

INPUT_CSV = "vending-machines-formateado.csv"
DATABASE_NAME = "vending_api_data.db"

# Continuar desde el registro 233.
# Esto omite los primeros 232 registros ya procesados.
START_ROW = 434

# Si quieres probar solo pocas filas desde START_ROW, coloca un número, por ejemplo 3.
# Para procesar desde START_ROW hasta el final, déjalo como None.
RUN_LIMIT = None

# Evita duplicar registros si por error vuelves a ejecutar una parte ya guardada.
SKIP_ALREADY_SAVED = True

# Pausa entre registros para evitar rate limits.
SLEEP_SECONDS_BETWEEN_ROWS = 1.5

# Inicializar cliente Firecrawl
firecrawl_client = Firecrawl(api_key=FIRECRAWL_API_KEY)


# ------------------------------------------------------------
# 3. VALIDACIONES INICIALES
# ------------------------------------------------------------

def validate_config():
    if not SERPER_API_KEY or SERPER_API_KEY == "PEGA_AQUI_TU_SERPER_API_KEY":
        raise ValueError("Debes pegar tu SERPER_API_KEY real en la variable SERPER_API_KEY.")

    if not FIRECRAWL_API_KEY or FIRECRAWL_API_KEY == "PEGA_AQUI_TU_FIRECRAWL_API_KEY":
        raise ValueError("Debes pegar tu FIRECRAWL_API_KEY real en la variable FIRECRAWL_API_KEY.")

    if not os.path.exists(INPUT_CSV):
        raise FileNotFoundError(
            f"No se encontró el archivo '{INPUT_CSV}'. "
            f"Sube el CSV a Colab o cambia INPUT_CSV con el nombre correcto."
        )


# ------------------------------------------------------------
# 4. ESQUEMAS PYDANTIC
# ------------------------------------------------------------

class PhysicalSpecs(BaseModel):
    alto_mm: Optional[int] = Field(None, description="Altura física del equipo en milímetros")
    ancho_mm: Optional[int] = Field(None, description="Ancho físico del equipo en milímetros")
    profundidad_mm: Optional[int] = Field(None, description="Profundidad física del equipo en milímetros")
    peso_kg: Optional[int] = Field(None, description="Peso neto del equipo vacío en kilogramos")


class EnergySpecs(BaseModel):
    voltaje: Optional[str] = Field(None, description="Voltaje eléctrico requerido, por ejemplo 230V o 110V")
    potencia_watts: Optional[int] = Field(None, description="Consumo eléctrico nominal máximo en Watts")
    gas_refrigerante: Optional[str] = Field(None, description="Tipo de gas refrigerante si aplica, por ejemplo R290 o R134a")


class HardwareSpecs(BaseModel):
    grupo_infusor: Optional[str] = Field(None, description="Modelo o patente del grupo infusor de café si aplica")
    mecanica_extraccion: Optional[str] = Field(None, description="Tipo de dispensación física, por ejemplo espirales, cintas, ascensor o vaso")
    capacidad_vasos: Optional[int] = Field(None, description="Capacidad total de vasos si es máquina de café")
    capacidad_canales_o_espirales: Optional[str] = Field(None, description="Número de selecciones, canales, bandejas o espirales")
    telemetria_integrada: Optional[bool] = Field(None, description="Indica si cuenta con telemetría, módem, nube o conectividad nativa")


class VendingMachineDetail(BaseModel):
    fabricante: Optional[str] = Field(None, description="Nombre del fabricante de la máquina")
    modelo_base: Optional[str] = Field(None, description="Modelo comercial de la máquina")
    tipo_maquina: Optional[str] = Field(
        None,
        description="Tipo de máquina: coffee, snack, drink, combo, food, frozen, locker u otro"
    )
    imagen_url: Optional[str] = Field(
        None,
        description="URL absoluta de la imagen principal, fotografía o render del modelo de máquina expendedora"
    )
    versiones_disponibles: List[str] = Field(
        default_factory=list,
        description="Otras subversiones o variantes del mismo modelo mencionadas"
    )
    especificaciones_fisicas: PhysicalSpecs = Field(default_factory=PhysicalSpecs)
    especificaciones_electricas: EnergySpecs = Field(default_factory=EnergySpecs)
    componentes_hardware: HardwareSpecs = Field(default_factory=HardwareSpecs)


# ------------------------------------------------------------
# 5. HELPERS GENERALES
# ------------------------------------------------------------

def safe_get(obj: Any, key: str, default=None):
    if obj is None:
        return default

    if isinstance(obj, dict):
        return obj.get(key, default)

    return getattr(obj, key, default)


def to_plain_dict(obj: Any) -> Dict:
    if obj is None:
        return {}

    if isinstance(obj, dict):
        return obj

    if hasattr(obj, "model_dump"):
        return obj.model_dump()

    if hasattr(obj, "dict"):
        return obj.dict()

    if hasattr(obj, "__dict__"):
        return obj.__dict__

    return {}


def normalize_possible_json(json_data: Any) -> Optional[dict]:
    if json_data is None:
        return None

    if hasattr(json_data, "model_dump"):
        json_data = json_data.model_dump()

    if hasattr(json_data, "dict"):
        json_data = json_data.dict()

    if isinstance(json_data, str):
        try:
            json_data = json.loads(json_data)
        except Exception:
            return None

    if not isinstance(json_data, dict):
        return None

    return json_data


def infer_tipo_maquina(text: str) -> Optional[str]:
    if not text:
        return None

    t = text.lower()

    if any(x in t for x in ["coffee", "café", "cafe", "espresso", "capuccino", "cappuccino", "vitro"]):
        return "coffee"

    if any(x in t for x in ["snack", "spiral", "espiral", "chips", "chocolate", "candy", "dulces"]):
        return "snack"

    if any(x in t for x in ["drink", "beverage", "bebida", "soda", "can", "bottle", "lata", "botella"]):
        return "drink"

    if any(x in t for x in ["combo", "snack and drink", "snacks and drinks"]):
        return "combo"

    if any(x in t for x in ["frozen", "ice cream", "helado", "congelado"]):
        return "frozen"

    if any(x in t for x in ["locker", "smart locker"]):
        return "locker"

    if any(x in t for x in ["food", "fresh food", "comida", "meal"]):
        return "food"

    return None


def is_probably_bad_image(url: str) -> bool:
    if not url:
        return True

    lowered = url.lower()

    if lowered.startswith("data:"):
        return True

    bad_keywords = [
        "logo",
        "favicon",
        "icon",
        "sprite",
        "placeholder",
        "blank",
        "spinner",
        "loading",
        "whatsapp",
        "facebook",
        "instagram",
        "linkedin",
        "youtube",
        "twitter",
        "x-twitter"
    ]

    if any(bad in lowered for bad in bad_keywords):
        return True

    bad_extensions = [".svg", ".ico"]

    if any(lowered.endswith(ext) for ext in bad_extensions):
        return True

    return False


def score_image_url(img_url: str, fabricante: str, modelo: str) -> int:
    lowered = img_url.lower()

    fabricante_words = [
        x.lower()
        for x in fabricante.replace("-", " ").split()
        if len(x) > 2
    ]

    modelo_words = [
        x.lower()
        for x in modelo.replace("-", " ").split()
        if len(x) > 1
    ]

    score = 0

    for word in fabricante_words:
        if word in lowered:
            score += 4

    for word in modelo_words:
        if word in lowered:
            score += 5

    product_keywords = [
        "product",
        "producto",
        "machine",
        "maquina",
        "máquina",
        "vending",
        "catalog",
        "catalogo",
        "render",
        "photo",
        "image",
        "vitro",
        "novara",
        "palma",
        "snack",
        "coffee",
        "cafe",
        "drink"
    ]

    for keyword in product_keywords:
        if keyword in lowered:
            score += 2

    good_extensions = [".jpg", ".jpeg", ".png", ".webp"]

    if any(ext in lowered for ext in good_extensions):
        score += 2

    return score


def extract_images_from_html(
    html: Optional[str],
    base_url: str,
    fabricante: str,
    modelo: str
) -> List[str]:
    if not html:
        return []

    soup = BeautifulSoup(html, "html.parser")
    images = []

    for img in soup.find_all("img"):
        candidates = [
            img.get("src"),
            img.get("data-src"),
            img.get("data-lazy-src"),
            img.get("data-original"),
        ]

        srcset = img.get("srcset")

        if srcset:
            parts = [
                p.strip().split(" ")[0]
                for p in srcset.split(",")
                if p.strip()
            ]
            candidates.extend(parts)

        for src in candidates:
            if not src:
                continue

            absolute_url = urljoin(base_url, src)

            if is_probably_bad_image(absolute_url):
                continue

            images.append(absolute_url)

    seen = set()
    unique_images = []

    for img_url in images:
        if img_url not in seen:
            seen.add(img_url)
            unique_images.append(img_url)

    unique_images.sort(
        key=lambda x: score_image_url(x, fabricante, modelo),
        reverse=True
    )

    return unique_images


def choose_best_image_from_html(
    html: Optional[str],
    base_url: str,
    fabricante: str,
    modelo: str
) -> Optional[str]:
    images = extract_images_from_html(html, base_url, fabricante, modelo)

    if images:
        return images[0]

    return None


# ------------------------------------------------------------
# 6. BASE DE DATOS SQLITE
# ------------------------------------------------------------

def init_db():
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS catalogo_vending (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            fabricante TEXT,
            modelo TEXT,
            url_oficial TEXT,
            datos_json TEXT,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
        )
    """)

    cursor.execute("PRAGMA table_info(catalogo_vending)")
    existing_columns = [column[1] for column in cursor.fetchall()]

    if "tipo_maquina" not in existing_columns:
        cursor.execute("""
            ALTER TABLE catalogo_vending
            ADD COLUMN tipo_maquina TEXT
        """)
        print("[+] Columna agregada: tipo_maquina")

    if "imagen_url" not in existing_columns:
        cursor.execute("""
            ALTER TABLE catalogo_vending
            ADD COLUMN imagen_url TEXT
        """)
        print("[+] Columna agregada: imagen_url")

    cursor.execute("""
        CREATE INDEX IF NOT EXISTS idx_catalogo_vending_fabricante_modelo
        ON catalogo_vending (fabricante, modelo)
    """)

    conn.commit()
    return conn


def record_exists(conn, fabricante: str, modelo: str, url_oficial: str) -> bool:
    cursor = conn.cursor()

    cursor.execute("""
        SELECT COUNT(*)
        FROM catalogo_vending
        WHERE fabricante = ?
          AND modelo = ?
          AND url_oficial = ?
    """, (fabricante, modelo, url_oficial))

    count = cursor.fetchone()[0]

    return count > 0


def save_to_db(
    conn,
    fabricante: str,
    modelo: str,
    url_oficial: str,
    datos_tecnicos: dict
):
    cursor = conn.cursor()

    tipo_maquina = datos_tecnicos.get("tipo_maquina")
    imagen_url = datos_tecnicos.get("imagen_url")

    cursor.execute("""
        INSERT INTO catalogo_vending (
            fabricante,
            modelo,
            url_oficial,
            tipo_maquina,
            imagen_url,
            datos_json
        )
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        fabricante,
        modelo,
        url_oficial,
        tipo_maquina,
        imagen_url,
        json.dumps(datos_tecnicos, ensure_ascii=False)
    ))

    conn.commit()


# ------------------------------------------------------------
# 7. BÚSQUEDA CON SERPER
# ------------------------------------------------------------

def search_url_with_serper(fabricante: str, modelo: str) -> Optional[str]:
    url = "https://google.serper.dev/search"

    query = f'"{fabricante}" "{modelo}" vending machine technical specifications official'

    payload = {
        "q": query,
        "gl": "us",
        "hl": "en",
        "num": 5
    }

    headers = {
        "X-API-KEY": SERPER_API_KEY,
        "Content-Type": "application/json"
    }

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=20)
        response.raise_for_status()

        results = response.json()
        organic_results = results.get("organic", [])

        if not organic_results:
            return None

        fabricante_clean = fabricante.lower().replace(" ", "")
        scored_results = []

        for item in organic_results:
            link = item.get("link")
            title = item.get("title", "")
            snippet = item.get("snippet", "")

            if not link:
                continue

            score = 0
            combined = f"{link} {title} {snippet}".lower()

            if fabricante.lower() in combined:
                score += 5

            if modelo.lower() in combined:
                score += 5

            if fabricante_clean in link.lower().replace("-", "").replace("_", ""):
                score += 5

            if any(x in combined for x in [
                "official",
                "specification",
                "technical",
                "datasheet",
                "brochure",
                "product"
            ]):
                score += 2

            scored_results.append((score, link))

        if not scored_results:
            return organic_results[0].get("link")

        scored_results.sort(reverse=True, key=lambda x: x[0])

        return scored_results[0][1]

    except Exception as e:
        print(f"  [-] Error al consultar Serper.dev para {fabricante} {modelo}: {e}")
        return None


# ------------------------------------------------------------
# 8. SCRAPER CON FIRECRAWL
# ------------------------------------------------------------

def scrape_with_firecrawl(url: str, fabricante: str, modelo: str) -> Optional[dict]:
    try:
        prompt = f"""
Extrae información técnica únicamente del siguiente modelo de máquina expendedora.

Fabricante esperado: {fabricante}
Modelo esperado: {modelo}

Reglas estrictas:
- Si la página contiene varios modelos, enfócate solo en el modelo indicado.
- No inventes datos técnicos que no aparezcan en la página.
- Si un valor no existe o no es seguro, usa null.
- Convierte medidas físicas a milímetros cuando sea posible.
- Convierte peso a kilogramos cuando sea posible.
- La imagen_url debe ser una URL absoluta de una fotografía o render de la máquina.
- No uses logos, favicons, íconos de redes sociales ni imágenes decorativas.
- Para tipo_maquina usa uno de estos valores si aplica: coffee, snack, drink, combo, food, frozen, locker.
"""

        schema = VendingMachineDetail.model_json_schema()

        scrape_result = firecrawl_client.scrape(
            url,
            formats=[
                "markdown",
                "html",
                {
                    "type": "json",
                    "schema": schema,
                    "prompt": prompt
                }
            ],
            only_main_content=False,
            wait_for=3000,
            timeout=120000
        )

        json_data = safe_get(scrape_result, "json")
        html_data = safe_get(scrape_result, "html")
        markdown_data = safe_get(scrape_result, "markdown")

        json_data = normalize_possible_json(json_data)

        if not json_data:
            print("  [-] Firecrawl respondió, pero no devolvió JSON estructurado.")
            return None

        json_data["fabricante"] = json_data.get("fabricante") or fabricante
        json_data["modelo_base"] = json_data.get("modelo_base") or modelo

        if not json_data.get("tipo_maquina"):
            combined_text = f"{fabricante} {modelo} {markdown_data or ''}"
            json_data["tipo_maquina"] = infer_tipo_maquina(combined_text)

        current_image = json_data.get("imagen_url")

        if current_image:
            current_image = urljoin(url, current_image)

        if not current_image or is_probably_bad_image(current_image):
            fallback_image = choose_best_image_from_html(
                html=html_data,
                base_url=url,
                fabricante=fabricante,
                modelo=modelo
            )
            json_data["imagen_url"] = fallback_image
        else:
            json_data["imagen_url"] = current_image

        json_data.setdefault("versiones_disponibles", [])
        json_data.setdefault("especificaciones_fisicas", {})
        json_data.setdefault("especificaciones_electricas", {})
        json_data.setdefault("componentes_hardware", {})

        return json_data

    except TypeError as e:
        print(f"  [-] Error de compatibilidad con SDK Firecrawl: {e}")
        print("      Reinstala en Colab con: !pip uninstall -y firecrawl firecrawl-py && !pip install -U firecrawl-py")
        return None

    except Exception as e:
        print(f"  [-] Error en Firecrawl al procesar {url}: {e}")
        return None


# ------------------------------------------------------------
# 9. LECTURA Y VALIDACIÓN DEL CSV
# ------------------------------------------------------------

def load_csv_rows(input_csv: str) -> List[dict]:
    df = pd.read_csv(input_csv)

    required_columns = ["Fabricante", "Modelo"]

    for col in required_columns:
        if col not in df.columns:
            raise ValueError(f"El CSV debe tener la columna obligatoria: {col}")

    if "URL" not in df.columns:
        df["URL"] = ""

    df = df.fillna("")

    rows = df.to_dict(orient="records")

    return rows


# ------------------------------------------------------------
# 10. PIPELINE PRINCIPAL
# ------------------------------------------------------------

def main():
    validate_config()

    print("[*] Iniciando pipeline de extracción técnica con imágenes...")
    print(f"[*] CSV de entrada: {INPUT_CSV}")
    print(f"[*] Base SQLite: {DATABASE_NAME}")
    print(f"[*] Inicio configurado desde fila: {START_ROW}")

    all_rows = load_csv_rows(INPUT_CSV)
    total_original = len(all_rows)

    if START_ROW < 1:
        raise ValueError("START_ROW debe ser mayor o igual a 1.")

    if START_ROW > total_original:
        raise ValueError(
            f"START_ROW={START_ROW} es mayor que el total de filas del CSV: {total_original}."
        )

    start_index = START_ROW - 1
    rows = all_rows[start_index:]

    if RUN_LIMIT is not None:
        rows = rows[:RUN_LIMIT]
        print(
            f"[*] Modo prueba activo. Se procesarán {RUN_LIMIT} filas "
            f"desde el registro {START_ROW}."
        )
    else:
        print(
            f"[*] Se procesarán {len(rows)} filas, "
            f"desde el registro {START_ROW} hasta el {total_original}."
        )

    db_conn = init_db()

    total_to_try = len(rows)
    saved = 0
    failed = 0
    skipped = 0

    for offset, row in enumerate(rows):
        real_index = START_ROW + offset

        fabricante = str(row.get("Fabricante", "")).strip()
        modelo = str(row.get("Modelo", "")).strip()
        url_csv = str(row.get("URL", "")).strip()

        if not fabricante or not modelo:
            print(f"\n[{real_index}/{total_original}] Registro incompleto. Omitiendo fila.")
            failed += 1
            continue

        print(f"\n[{real_index}/{total_original}] Procesando: {fabricante} - {modelo}")

        url_oficial = url_csv

        if not url_oficial:
            print("  [*] URL no provista. Buscando con Serper.dev...")
            url_oficial = search_url_with_serper(fabricante, modelo)

            if not url_oficial:
                print(f"  [-] No se pudo descubrir URL para: {fabricante} - {modelo}")
                failed += 1
                continue

            print(f"  [+] URL descubierta: {url_oficial}")
        else:
            print(f"  [+] Usando URL del CSV: {url_oficial}")

        if SKIP_ALREADY_SAVED and record_exists(db_conn, fabricante, modelo, url_oficial):
            print("  [>] Registro ya existe en la base. Omitiendo para evitar duplicado.")
            skipped += 1
            continue

        print("  [*] Extrayendo datos técnicos e imagen con Firecrawl...")
        datos_tecnicos = scrape_with_firecrawl(url_oficial, fabricante, modelo)

        if not datos_tecnicos:
            print(f"  [-] No se pudieron extraer datos válidos de: {url_oficial}")
            failed += 1
            continue

        save_to_db(
            conn=db_conn,
            fabricante=fabricante,
            modelo=modelo,
            url_oficial=url_oficial,
            datos_tecnicos=datos_tecnicos
        )

        saved += 1

        print("  [✓] Registro guardado correctamente.")
        print(f"      Tipo: {datos_tecnicos.get('tipo_maquina')}")
        print(f"      Imagen: {datos_tecnicos.get('imagen_url')}")

        time.sleep(SLEEP_SECONDS_BETWEEN_ROWS)

    db_conn.close()

    print("\n============================================================")
    print("[✓] Pipeline finalizado")
    print(f"    Total filas del CSV: {total_original}")
    print(f"    Inicio configurado: fila {START_ROW}")
    print(f"    Filas intentadas en esta ejecución: {total_to_try}")
    print(f"    Guardados en esta ejecución: {saved}")
    print(f"    Omitidos por duplicado: {skipped}")
    print(f"    Fallidos/Omitidos por error: {failed}")
    print(f"    Base generada/actualizada: {DATABASE_NAME}")
    print("============================================================")


# ------------------------------------------------------------
# 11. EJECUCIÓN
# ------------------------------------------------------------

if __name__ == "__main__":
    main()

[*] Iniciando pipeline de extracción técnica con imágenes...
[*] CSV de entrada: vending-machines-formateado.csv
[*] Base SQLite: vending_api_data.db
[*] Inicio configurado desde fila: 434
[*] Se procesarán 113 filas, desde el registro 434 hasta el 546.

[434/546] Procesando: XY Vending - Nail Sticker Vending Machine
  [+] Usando URL del CSV: https://www.xyvend.com/products/
  [*] Extrayendo datos técnicos e imagen con Firecrawl...
  [✓] Registro guardado correctamente.
      Tipo: coffee
      Imagen: https://www.xyvend.com/products/makeup-vending-machine/nail-sticker-vending-machine/

[435/546] Procesando: Micron Vending - WM22 snack drink
  [+] Usando URL del CSV: https://www.micronvending.com/PRODUCTS-micron-smart-vending.html
  [*] Extrayendo datos técnicos e imagen con Firecrawl...
  [✓] Registro guardado correctamente.
      Tipo: snack
      Imagen: https://www.micronvending.com/egg-locker-vending-machine-card-reader-farm-produce-high-tach-vending-machine..html

[436/546] Proce